# Shield-Fi: Fraud Detection — Exploration & Explainability

This notebook covers:
1. Data exploration & class imbalance visualization
2. Feature distribution (fraud vs legit)
3. SHAP value analysis — why did the model flag this transaction?
4. Threshold tuning — PR curve analysis

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import joblib
from pathlib import Path
from sklearn.metrics import precision_recall_curve, average_precision_score
from features import FraudFeatureEngineer, get_feature_names

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Load Data & Check Class Imbalance

In [ ]:
df = pd.read_csv('../data/transactions.csv')
print(f'Shape: {df.shape}')
print(f'Fraud rate: {df["is_fraud"].mean()*100:.3f}%')
df.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Class distribution
counts = df['is_fraud'].value_counts()
axes[0].bar(['Legit', 'Fraud'], counts.values, color=['#2ecc71','#e74c3c'])
axes[0].set_title('Class Distribution (raw)')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v+200, f'{v:,}', ha='center')

# Amount distribution
df[df['is_fraud']==0]['amount'].clip(upper=1000).plot.hist(ax=axes[1], bins=50, alpha=0.6, color='#2ecc71', label='Legit')
df[df['is_fraud']==1]['amount'].clip(upper=1000).plot.hist(ax=axes[1], bins=50, alpha=0.6, color='#e74c3c', label='Fraud')
axes[1].set_title('Amount Distribution (clipped at $1000)')
axes[1].legend()
plt.tight_layout()

## 2. Feature Engineering Preview

In [ ]:
X_raw = df.drop(columns=['is_fraud', 'transaction_id'])
y     = df['is_fraud'].values

eng = FraudFeatureEngineer()
eng.fit(X_raw, y)
X = eng.transform(X_raw)
feature_names = get_feature_names()

feat_df = pd.DataFrame(X, columns=feature_names)
feat_df['is_fraud'] = y
feat_df.head()

In [ ]:
# Compare velocity features: fraud vs legit
vel_features = ['velocity_count_300s', 'velocity_count_3600s', 'velocity_count_86400s']
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, feat in zip(axes, vel_features):
    feat_df.boxplot(column=feat, by='is_fraud', ax=ax)
    ax.set_title(feat)
    ax.set_xlabel('is_fraud')
plt.suptitle('Velocity Features: Fraud vs Legit')
plt.tight_layout()

## 3. Load Trained Model & SHAP Analysis

In [ ]:
model = joblib.load('../models/xgb_model.joblib')
print('Model loaded. Best iteration:', model.best_iteration)

In [ ]:
# SHAP on a sample (200 rows for speed)
np.random.seed(42)
sample_idx = np.where(y == 1)[0][:100]   # 100 fraud cases
legit_idx  = np.where(y == 0)[0][:100]   # 100 legit cases
idx        = np.concatenate([sample_idx, legit_idx])
X_sample   = X[idx]

explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_sample)

shap.summary_plot(shap_values, X_sample, feature_names=feature_names, max_display=15)

In [ ]:
# Force plot for a single fraud transaction
fraud_local = X[sample_idx[0]:sample_idx[0]+1]
shap.force_plot(
    explainer.expected_value,
    explainer.shap_values(fraud_local)[0],
    fraud_local[0],
    feature_names=feature_names,
    matplotlib=True
)

## 4. Threshold Tuning

In [ ]:
from sklearn.model_selection import train_test_split

_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
y_prob = model.predict_proba(X_test)[:,1]

prec, rec, thresholds = precision_recall_curve(y_test, y_prob)
f1 = 2 * prec * rec / (prec + rec + 1e-9)

best_thresh = thresholds[np.argmax(f1)]
print(f'Best threshold (F1): {best_thresh:.3f}')

plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(thresholds, prec[:-1], label='Precision')
plt.plot(thresholds, rec[:-1],  label='Recall')
plt.plot(thresholds, f1[:-1],   label='F1', lw=2)
plt.axvline(best_thresh, ls='--', color='grey', label=f'Best={best_thresh:.2f}')
plt.xlabel('Threshold'); plt.legend(); plt.title('Precision/Recall/F1 vs Threshold')

plt.subplot(1,2,2)
pr_auc = average_precision_score(y_test, y_prob)
plt.plot(rec, prec, color='#e74c3c', lw=2)
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title(f'PR Curve — AUC={pr_auc:.4f}')
plt.tight_layout()